In [52]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hocop1/kitti-odometry/poses/06.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/10.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/05.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/01.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/08.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/07.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/03.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/00.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/02.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/09.txt
/kaggle/input/datasets/hocop1/kitti-odometry/poses/04.txt
/kaggle/input/datasets/hocop1/kitti-odometry/sequences/17/calib.txt
/kaggle/input/datasets/hocop1/kitti-odometry/sequences/17/times.txt
/kaggle/input/datasets/hocop1/kitti-odometry/sequences/17/image_3/000419.png
/kaggle/input/datasets/hocop1/kitti-odometry/sequences/17/image_3/000313.png
/kaggle/input/datasets/hocop1/kitti-odometry/sequences/17/image_3/000224

In [70]:
import os
import glob
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

def load_pose_line(line):
    vals = np.fromstring(line, sep=' ')
    T = np.eye(4, dtype=np.float32)
    T[:3, :4] = vals.reshape(3, 4)
    return T

class KITTIOdometryDataset(Dataset):
    def __init__(self, root, sequence="00", image_folder="image_2", transform=None):
        self.root = root
        self.sequence = sequence
        self.transform = transform

        seq_dir = os.path.join(root, "sequences", sequence, image_folder)
        pose_file = os.path.join(root, "poses", f"{sequence}.txt")

        self.image_paths = sorted(glob.glob(os.path.join(seq_dir, "*.png")))

        with open(pose_file, "r") as f:
            self.poses = [load_pose_line(line.strip()) for line in f.readlines()]

        assert len(self.image_paths) == len(self.poses)

    def __len__(self):
        return len(self.image_paths) - 1

    def __getitem__(self, idx):
        img1 = Image.open(self.image_paths[idx]).convert("RGB")
        img2 = Image.open(self.image_paths[idx + 1]).convert("RGB")
    
        T1 = self.poses[idx]
        T2 = self.poses[idx + 1]
        T_rel = np.linalg.inv(T1) @ T2
    
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
    
        target = torch.tensor(T_rel[:3, :4].reshape(-1), dtype=torch.float32)
    
        return {
            "img1": img1,
            "img2": img2,
            "target": target
        }

In [54]:
import torchvision.transforms as Transforms 
transform = Transforms.Compose([Transforms.ToTensor(), Transforms.Resize((384 ,1248))])
dataset = KITTIOdometryDataset(root="/kaggle/input/datasets/hocop1/kitti-odometry/", transform=transform)

In [55]:
from torch.utils.data import DataLoader

dataset = KITTIOdometryDataset(root="/kaggle/input/datasets/hocop1/kitti-odometry/", transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [56]:
IMAGE_X= 384
IMAGE_Y = 1248
CHANNELS = 3

PATCH_SIZE=16
EMBED_DIM=8

In [61]:
import torch.nn as nn
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, EMBED_DIM,PATCH_SIZE,PATCH_SIZE)
    def forward(self, x):
        x = self.conv(x)
        x = x.flatten(2).permute(0,2,1)
        return x



In [71]:
import torch
import torch.nn as nn

EMBED_DIM = 128
PATCH_SIZE = 16
MLP_NODES = 128
OUT_DIM = 12   # or 6

class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)

    def forward(self, x):
        x = self.conv(x)                      # [B, D, H/P, W/P]
        x = x.flatten(2).permute(0, 2, 1)    # [B, N, D]
        return x

class TransformEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ImageEncoder()
        self.head = nn.Sequential(
            nn.Linear(EMBED_DIM * 3, MLP_NODES),
            nn.GELU(),
            nn.Linear(MLP_NODES, MLP_NODES),
            nn.GELU(),
            nn.Linear(MLP_NODES, OUT_DIM)
        )

    def forward(self, x):
        img1 = x["img1"]
        img2 = x["img2"]

        z1 = self.encoder(img1)                  # [B, N, D]
        z2 = self.encoder(img2)                  # [B, N, D]

        z = torch.cat([z1, z2, z2 - z1], dim=-1)  # [B, N, 3D]
        z = z.mean(dim=1)                          # [B, 3D]

        return self.head(z)                        # [B, OUT_DIM]

In [59]:
data = next(iter(loader))
conv = nn.Conv2d(3, EMBED_DIM,PATCH_SIZE,PATCH_SIZE)
print(conv(data['img1']).flatten(2).permute(0,2,1).shape)

torch.Size([8, 1872, 8])


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# full dataset
full_dataset = KITTIOdometryDataset(root="/kaggle/input/datasets/hocop1/kitti-odometry/", sequence="00", image_folder="image_2", transform=transform)

# split sizes
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

model = TransformEstimator().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        img1 = batch["img1"].to(device)
        img2 = batch["img2"].to(device)
        target = batch["target"].to(device)   # [B, 12]

        optimizer.zero_grad()

        pred = model({
            "img1": img1,
            "img2": img2
        })

        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for batch in loader:
        img1 = batch["img1"].to(device)
        img2 = batch["img2"].to(device)
        target = batch["target"].to(device)

        pred = model({
            "img1": img1,
            "img2": img2
        })

        loss = criterion(pred, target)
        total_loss += loss.item()

    return total_loss / len(loader)

EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

Epoch 1/10 | Train Loss: 0.012482 | Test Loss: 0.007110
Epoch 2/10 | Train Loss: 0.006597 | Test Loss: 0.006706
Epoch 3/10 | Train Loss: 0.006410 | Test Loss: 0.007038
Epoch 4/10 | Train Loss: 0.006376 | Test Loss: 0.006543
Epoch 5/10 | Train Loss: 0.006370 | Test Loss: 0.006534
Epoch 6/10 | Train Loss: 0.006339 | Test Loss: 0.006412
